# Protein Regression: Two-source Mixed Holdout Validation

This notebook builds protein regression models from the combined 2024 and 2025 sample pool.

- Fixed model development set: 89 samples from 2024 + 24 samples from 2025 = 113 samples
- Independent validation set: 24 samples from 2024 + 12 samples from 2025 = 36 samples
- The 2025 validation subset is balanced across the three treatments.
- Repeated two-source holdout validation is retained as an optional analysis and is disabled by default.


In [ ]:
# -*- coding: utf-8 -*-

# ============================================================
# Protein regression - two-source mixed holdout validation
#
# Source A: 2024 samples (113)
# Source B: 2025 samples (36)
#
# Fixed split:
#   Training:   A 89 + B 24 = 113 samples
#   Validation: A 24 + B 12 = 36 samples
#
# Repeated validation:
#   Repeat the same split strategy 50 times.
#
# Target:
#   Protein content
#
# Modeling:
#   Patch-level spectra are used for model fitting.
#   Patch-level predictions are averaged to sample-level predictions.
# ============================================================

import os
import re
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from IPython.display import display

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


# ============================================================
# 0. Paths
# ============================================================

def find_project_root(start_path=None):
    """
    Find the repository root by searching the current directory and its
    parents for the required 'Pea samples-new' data directory.
    """
    current = Path(start_path or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "Pea samples-new").is_dir():
            return candidate

    raise FileNotFoundError(
        "Cannot locate the project root. Run this notebook from inside the "
        "GitHub repository and make sure the repository contains the "
        "'Pea samples-new' directory."
    )


PROJECT_DIR = find_project_root()
DATA_BASE = PROJECT_DIR / "Pea samples-new"
MODEL_DIR = DATA_BASE / "Regression model"

# 2024 patch dataset
A_PATCH_CSV = DATA_BASE / "pea_patch_dataset.csv"

# 2025 patch dataset and measured quality data
B_PATCH_CSV = MODEL_DIR / "flour_external_pea_patch_dataset.csv"
B_QUALITY_XLSX = MODEL_DIR / "pea quality-2025.xlsx"

required_input_files = [
    A_PATCH_CSV,
    B_PATCH_CSV,
    B_QUALITY_XLSX,
]

missing_input_files = [path for path in required_input_files if not path.is_file()]
if missing_input_files:
    missing_text = "\n".join(f"- {path}" for path in missing_input_files)
    raise FileNotFoundError(
        "The following required input files were not found:\n"
        f"{missing_text}"
    )

OUT_DIR = PROJECT_DIR / "Pea_TwoSource_Mixed113_ProteinRegression_Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

REPEAT_OUT_DIR = PROJECT_DIR / "Pea_TwoSource_Mixed113_ProteinRegression_RepeatedValidation"
REPEAT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_DIR)
print("2024 patch CSV:", A_PATCH_CSV)
print("2025 patch CSV:", B_PATCH_CSV)
print("2025 quality Excel:", B_QUALITY_XLSX)
print("Fixed model output folder:", OUT_DIR)
print("Repeated validation output folder:", REPEAT_OUT_DIR)


# ============================================================
# 1. Target variable
# ============================================================

TARGET_COL = "Protein content"

print("\nRegression target:", TARGET_COL)


# ============================================================
# 2. Helper functions
# ============================================================

def normalize_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).lower())


def detect_sample_id_column(df_input):
    candidates = [
        "sample_id", "Sample ID", "Sample_ID", "sample", "Sample",
        "ID", "id", "No.", "No"
    ]

    norm_map = {normalize_name(c): c for c in df_input.columns}

    for cand in candidates:
        key = normalize_name(cand)
        if key in norm_map:
            return norm_map[key]

    raise ValueError("Cannot detect sample ID column.")


def detect_wavelength_columns(df, expected_min=900, expected_max=2600):
    pairs = []

    for c in df.columns:
        nums = re.findall(r"\d+\.\d+|\d+", str(c))
        if nums:
            try:
                val = float(nums[-1])
                if expected_min <= val <= expected_max:
                    pairs.append((c, val))
            except Exception:
                pass

    pairs = sorted(pairs, key=lambda x: x[1])

    wl_cols = [x[0] for x in pairs]
    wavelengths = np.array([x[1] for x in pairs], dtype=float)

    return wl_cols, wavelengths


def align_external_to_training_wavelengths(ext_df, train_wl_cols, train_wavelengths):
    ext_wl_cols, ext_wavelengths = detect_wavelength_columns(ext_df)

    ext_map = {
        round(float(wl), 2): col
        for col, wl in zip(ext_wl_cols, ext_wavelengths)
    }

    aligned_ext_cols = []
    missing = []

    for train_col, train_wl in zip(train_wl_cols, train_wavelengths):
        key = round(float(train_wl), 2)

        if key in ext_map:
            aligned_ext_cols.append(ext_map[key])
        else:
            missing.append(train_wl)

    if len(missing) > 0:
        raise ValueError(
            f"Source B spectra are missing {len(missing)} Source A training wavelengths.\n"
            f"First missing wavelengths: {missing[:20]}\n"
            "You need wavelength alignment/interpolation before modeling."
        )

    return aligned_ext_cols


QUALITY_CANDIDATES = {
    "Protein content": [
        "Protein content", "protein content", "Protein", "protein"
    ],
}


def detect_quality_columns(df, required_quality):
    norm_map = {normalize_name(c): c for c in df.columns}
    found = {}

    for q in required_quality:
        actual = None

        for cand in QUALITY_CANDIDATES[q]:
            key = normalize_name(cand)
            if key in norm_map:
                actual = norm_map[key]
                break

        if actual is not None:
            found[q] = actual

    return found


def load_quality_excel(xlsx_path, required_quality):
    if not xlsx_path.exists():
        raise FileNotFoundError(f"Quality Excel file not found: {xlsx_path}")

    xls = pd.ExcelFile(xlsx_path)

    for sheet in xls.sheet_names:
        temp = pd.read_excel(xlsx_path, sheet_name=sheet)
        qmap = detect_quality_columns(temp, required_quality)

        if all(q in qmap for q in required_quality):
            sid_col = detect_sample_id_column(temp)

            out = temp[[sid_col] + [qmap[q] for q in required_quality]].copy()
            out = out.rename(columns={sid_col: "sample_id"})

            rename_dict = {qmap[q]: q for q in required_quality}
            out = out.rename(columns=rename_dict)

            out["sample_id"] = pd.to_numeric(out["sample_id"], errors="coerce")
            out = out.dropna(subset=["sample_id"])
            out["sample_id"] = out["sample_id"].astype(int)

            out = out[["sample_id"] + required_quality]
            out = out.groupby("sample_id", as_index=False).median(numeric_only=True)

            return out

    raise ValueError(
        "Cannot find one sheet containing all required quality indicators:\n"
        + "\n".join(required_quality)
    )


def assign_b_treatment(sample_id):
    sid = int(sample_id)

    if 1 <= sid <= 12:
        return "Control"
    elif 13 <= sid <= 24:
        return "150% N"
    elif 25 <= sid <= 36:
        return "50% water"
    else:
        return "Unknown"


class SNVTransformer(BaseEstimator, TransformerMixin):
    """Standard Normal Variate correction applied row-wise."""
    def __init__(self, eps=1e-12):
        self.eps = eps

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        mean = np.mean(X, axis=1, keepdims=True)
        std = np.std(X, axis=1, keepdims=True)
        std = np.where(std < self.eps, 1.0, std)
        return (X - mean) / std


def aggregate_patch_predictions_to_sample(patch_df, y_patch_pred):
    temp = pd.DataFrame({
        "global_sample_id": patch_df["global_sample_id"].values,
        "source": patch_df["source"].values,
        "sample_id": patch_df["sample_id"].values,
        "treatment": patch_df["treatment"].values,
        "y_patch_pred": np.asarray(y_patch_pred).ravel(),
    })

    sample_pred = (
        temp
        .groupby(["global_sample_id", "source", "sample_id", "treatment"])["y_patch_pred"]
        .agg(["mean", "std", "count"])
        .reset_index()
        .rename(columns={
            "mean": "y_pred",
            "std": "patch_prediction_std",
            "count": "n_patches",
        })
    )

    sample_pred["patch_prediction_std"] = sample_pred["patch_prediction_std"].fillna(0)

    return sample_pred


def regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    bias = np.mean(y_pred - y_true)

    if rmse == 0:
        rpd = np.inf
    else:
        rpd = np.std(y_true, ddof=1) / rmse

    return {
        "n_samples": len(y_true),
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "rpd": rpd,
    }


def create_two_source_split_for_repeat(repeat_id, random_state=42):
    """
    Create one mixed-source split.

    Training:
        A 89 + B 24

    Validation:
        A 24 + B 12

    B validation is balanced by treatment:
        4 Control + 4 150% N + 4 50% water
    """

    rng = np.random.default_rng(random_state + repeat_id)

    a_ids = np.array(sorted(quality_a["global_sample_id"].unique()))

    if len(a_ids) != 113:
        raise ValueError(f"Source A should contain 113 samples, but got {len(a_ids)}.")

    a_val_ids = rng.choice(a_ids, size=24, replace=False)
    a_train_ids = np.array([x for x in a_ids if x not in a_val_ids])

    b_val_ids = []
    b_train_ids = []

    for treatment in ["Control", "150% N", "50% water"]:
        ids_t = np.array(sorted(
            quality_b.loc[
                quality_b["treatment"] == treatment,
                "global_sample_id"
            ].unique()
        ))

        if len(ids_t) != 12:
            raise ValueError(
                f"Source B treatment {treatment} should contain 12 samples, "
                f"but got {len(ids_t)}."
            )

        val_t = rng.choice(ids_t, size=4, replace=False)
        train_t = np.array([x for x in ids_t if x not in val_t])

        b_val_ids.extend(val_t.tolist())
        b_train_ids.extend(train_t.tolist())

    train_ids_repeat = sorted(a_train_ids.tolist() + b_train_ids)
    val_ids_repeat = sorted(a_val_ids.tolist() + b_val_ids)

    if len(train_ids_repeat) != 113:
        raise ValueError(f"Training set should contain 113 samples, but got {len(train_ids_repeat)}.")

    if len(val_ids_repeat) != 36:
        raise ValueError(f"Validation set should contain 36 samples, but got {len(val_ids_repeat)}.")

    return train_ids_repeat, val_ids_repeat


# ============================================================
# 3. Load Source A spectra and protein data
# ============================================================

df_a_raw = pd.read_csv(A_PATCH_CSV)

if "sample_id" not in df_a_raw.columns:
    sid_col = detect_sample_id_column(df_a_raw)
    df_a_raw = df_a_raw.rename(columns={sid_col: "sample_id"})

df_a_raw["sample_id"] = pd.to_numeric(df_a_raw["sample_id"], errors="coerce")
df_a_raw = df_a_raw.dropna(subset=["sample_id"]).copy()
df_a_raw["sample_id"] = df_a_raw["sample_id"].astype(int)

wl_cols, wavelengths = detect_wavelength_columns(df_a_raw)

if len(wl_cols) == 0:
    raise ValueError("No wavelength columns detected in Source A patch CSV.")

qmap_a = detect_quality_columns(df_a_raw, [TARGET_COL])

if TARGET_COL not in qmap_a:
    raise ValueError("Source A patch CSV is missing Protein content column.")

quality_a = (
    df_a_raw
    .groupby("sample_id")[[qmap_a[TARGET_COL]]]
    .median()
    .reset_index()
)

quality_a = quality_a.rename(columns={qmap_a[TARGET_COL]: TARGET_COL})
quality_a = quality_a.dropna(subset=[TARGET_COL]).reset_index(drop=True)

quality_a["source"] = "A_original"
quality_a["global_sample_id"] = quality_a["sample_id"].apply(lambda x: f"A_{int(x)}")
quality_a["treatment"] = "A_original"

df_a_spectra = df_a_raw[["sample_id"] + wl_cols].copy()
df_a_spectra["source"] = "A_original"
df_a_spectra["global_sample_id"] = df_a_spectra["sample_id"].apply(lambda x: f"A_{int(x)}")
df_a_spectra["treatment"] = "A_original"

df_a_spectra = df_a_spectra[
    ["global_sample_id", "source", "sample_id", "treatment"] + wl_cols
].copy()

quality_a = quality_a[
    ["global_sample_id", "source", "sample_id", "treatment", TARGET_COL]
].copy()

print("\nSource A:")
print("Patch rows:", df_a_spectra.shape[0])
print("Samples:", quality_a["global_sample_id"].nunique())
print("Wavelength columns:", len(wl_cols))
print("Protein range:")
print(quality_a[TARGET_COL].describe())
display(quality_a.head())


# ============================================================
# 4. Load Source B spectra and protein data
# ============================================================

if not B_PATCH_CSV.exists():
    raise FileNotFoundError(f"Source B patch CSV not found: {B_PATCH_CSV}")

df_b_raw = pd.read_csv(B_PATCH_CSV)

if "sample_id" not in df_b_raw.columns:
    sid_col = detect_sample_id_column(df_b_raw)
    df_b_raw = df_b_raw.rename(columns={sid_col: "sample_id"})

df_b_raw["sample_id"] = pd.to_numeric(df_b_raw["sample_id"], errors="coerce")
df_b_raw = df_b_raw.dropna(subset=["sample_id"]).copy()
df_b_raw["sample_id"] = df_b_raw["sample_id"].astype(int)

aligned_b_wl_cols = align_external_to_training_wavelengths(
    ext_df=df_b_raw,
    train_wl_cols=wl_cols,
    train_wavelengths=wavelengths,
)

df_b_spectra = df_b_raw[["sample_id"] + aligned_b_wl_cols].copy()

rename_b_wl = {
    b_col: a_col
    for b_col, a_col in zip(aligned_b_wl_cols, wl_cols)
}

df_b_spectra = df_b_spectra.rename(columns=rename_b_wl)

quality_b = load_quality_excel(
    xlsx_path=B_QUALITY_XLSX,
    required_quality=[TARGET_COL],
)

quality_b = quality_b.dropna(subset=[TARGET_COL]).reset_index(drop=True)

quality_b["source"] = "B_second"
quality_b["global_sample_id"] = quality_b["sample_id"].apply(lambda x: f"B_{int(x)}")
quality_b["treatment"] = quality_b["sample_id"].apply(assign_b_treatment)

df_b_spectra["source"] = "B_second"
df_b_spectra["global_sample_id"] = df_b_spectra["sample_id"].apply(lambda x: f"B_{int(x)}")
df_b_spectra["treatment"] = df_b_spectra["sample_id"].apply(assign_b_treatment)

df_b_spectra = df_b_spectra[
    ["global_sample_id", "source", "sample_id", "treatment"] + wl_cols
].copy()

quality_b = quality_b[
    ["global_sample_id", "source", "sample_id", "treatment", TARGET_COL]
].copy()

print("\nSource B:")
print("Patch rows:", df_b_spectra.shape[0])
print("Samples:", quality_b["global_sample_id"].nunique())
print("Treatment distribution:")
print(quality_b["treatment"].value_counts())
print("Protein range:")
print(quality_b[TARGET_COL].describe())
display(quality_b.head())


# ============================================================
# 5. Combine A + B
# ============================================================

df_all_spectra = pd.concat(
    [df_a_spectra, df_b_spectra],
    axis=0,
    ignore_index=True
)

quality_all = pd.concat(
    [quality_a, quality_b],
    axis=0,
    ignore_index=True
)

if quality_all["global_sample_id"].duplicated().any():
    dup = quality_all.loc[
        quality_all["global_sample_id"].duplicated(),
        "global_sample_id"
    ].tolist()
    raise ValueError(f"Duplicated global_sample_id found: {dup[:10]}")

print("\nCombined A + B:")
print("Total samples:", quality_all["global_sample_id"].nunique())
print("A samples:", quality_all.query("source == 'A_original'")["global_sample_id"].nunique())
print("B samples:", quality_all.query("source == 'B_second'")["global_sample_id"].nunique())
print("Total patch rows:", df_all_spectra.shape[0])
print("Combined protein range:")
print(quality_all[TARGET_COL].describe())


# ============================================================
# 6. Define regression models
# ============================================================

models = {
    "PLSR_10": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", PLSRegression(
            n_components=10,
            scale=False,
        )),
    ]),

    "PLSR_15": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", PLSRegression(
            n_components=15,
            scale=False,
        )),
    ]),

    "SVR_RBF": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", SVR(
            kernel="rbf",
            C=10,
            gamma="scale",
            epsilon=0.1,
        )),
    ]),

    "RandomForest": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", RandomForestRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),

    "ExtraTrees": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", ExtraTreesRegressor(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=2,
            random_state=RANDOM_STATE,
            n_jobs=-1,
        )),
    ]),

    "Ridge": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", Ridge(
            alpha=10.0,
            random_state=RANDOM_STATE,
        )),
    ]),

    "ElasticNet": Pipeline([
        ("snv", SNVTransformer()),
        ("scaler", StandardScaler()),
        ("reg", ElasticNet(
            alpha=0.01,
            l1_ratio=0.2,
            max_iter=5000,
            random_state=RANDOM_STATE,
        )),
    ]),
}

print("\nRegression models:")
print(list(models.keys()))


# ============================================================
# 7. Fixed split: build original mixed-source 113 protein model
# ============================================================

train_ids, val_ids = create_two_source_split_for_repeat(
    repeat_id=0,
    random_state=RANDOM_STATE
)

split_df = pd.DataFrame({
    "global_sample_id": train_ids + val_ids,
    "split": ["train"] * len(train_ids) + ["validation"] * len(val_ids),
})

split_df = split_df.merge(
    quality_all[["global_sample_id", "source", "sample_id", "treatment", TARGET_COL]],
    on="global_sample_id",
    how="left"
)

split_df.to_csv(
    OUT_DIR / "fixed_split_train113_validation36_protein.csv",
    index=False
)

# Save the fixed sample IDs separately so that other notebooks can reproduce
# exactly the same 113/36 split without reconstructing it from predictions.
pd.DataFrame({
    "global_sample_id": train_ids,
}).to_csv(
    OUT_DIR / "fixed_training_sample_ids.csv",
    index=False
)

pd.DataFrame({
    "global_sample_id": val_ids,
}).to_csv(
    OUT_DIR / "fixed_validation_sample_ids.csv",
    index=False
)

print("\nFixed split summary:")
print(split_df.groupby(["split", "source"]).size())

print("\nFixed split B treatment distribution:")
print(split_df.query("source == 'B_second'").groupby(["split", "treatment"]).size())

print("\nProtein distribution by split:")
display(split_df.groupby("split")[TARGET_COL].describe())


# ============================================================
# 8. Prepare fixed training and validation patch data
# ============================================================

quality_train = quality_all[
    quality_all["global_sample_id"].isin(train_ids)
].copy()

quality_val = quality_all[
    quality_all["global_sample_id"].isin(val_ids)
].copy()

target_map_train = quality_train.set_index("global_sample_id")[TARGET_COL].to_dict()
target_map_val = quality_val.set_index("global_sample_id")[TARGET_COL].to_dict()

df_train_patches = df_all_spectra[
    df_all_spectra["global_sample_id"].isin(train_ids)
].copy()

df_val_patches = df_all_spectra[
    df_all_spectra["global_sample_id"].isin(val_ids)
].copy()

df_train_patches["y"] = df_train_patches["global_sample_id"].map(target_map_train).astype(float)
df_val_patches["y_true"] = df_val_patches["global_sample_id"].map(target_map_val).astype(float)

X_train_all = df_train_patches[wl_cols].values.astype(float)
y_train_all = df_train_patches["y"].values.astype(float)
groups_train = df_train_patches["global_sample_id"].values

X_val = df_val_patches[wl_cols].values.astype(float)

print("\nFixed training patch data:")
print("Training samples:", df_train_patches["global_sample_id"].nunique())
print("Training patches:", df_train_patches.shape[0])

print("\nFixed validation patch data:")
print("Validation samples:", df_val_patches["global_sample_id"].nunique())
print("Validation patches:", df_val_patches.shape[0])


# ============================================================
# 9. Internal GroupKFold on fixed training 113
# ============================================================

gkf = GroupKFold(n_splits=5)

internal_rows = []
oof_prediction_tables = {}

for model_name, model_template in models.items():
    print("\n" + "=" * 80)
    print("Internal GroupKFold regression model:", model_name)
    print("=" * 80)

    sample_pred_list = []

    for fold, (tr_idx, te_idx) in enumerate(
        gkf.split(X_train_all, y_train_all, groups=groups_train),
        start=1
    ):
        model = clone(model_template)

        X_tr, y_tr = X_train_all[tr_idx], y_train_all[tr_idx]
        X_te = X_train_all[te_idx]

        patch_test_df = df_train_patches.iloc[te_idx][
            ["global_sample_id", "source", "sample_id", "treatment"]
        ].copy()

        model.fit(X_tr, y_tr)

        y_patch_pred = model.predict(X_te)
        y_patch_pred = np.asarray(y_patch_pred).ravel()

        fold_sample_pred = aggregate_patch_predictions_to_sample(
            patch_df=patch_test_df,
            y_patch_pred=y_patch_pred,
        )

        true_map_fold = (
            df_train_patches
            .iloc[te_idx]
            .groupby("global_sample_id")["y"]
            .first()
            .to_dict()
        )

        fold_sample_pred["y_true"] = (
            fold_sample_pred["global_sample_id"]
            .map(true_map_fold)
            .astype(float)
        )

        fold_sample_pred["fold"] = fold
        fold_sample_pred["model"] = model_name

        sample_pred_list.append(fold_sample_pred)

    oof_pred = pd.concat(sample_pred_list, axis=0, ignore_index=True)

    oof_prediction_tables[model_name] = oof_pred

    metrics = regression_metrics(
        y_true=oof_pred["y_true"].values,
        y_pred=oof_pred["y_pred"].values,
    )

    metrics["model"] = model_name
    internal_rows.append(metrics)

    print(metrics)

internal_summary = pd.DataFrame(internal_rows)
internal_summary = internal_summary.sort_values(
    ["r2", "rpd"],
    ascending=False
).reset_index(drop=True)

print("\nInternal GroupKFold summary for fixed mixed-source training 113:")
display(internal_summary)

internal_summary.to_csv(
    OUT_DIR / "internal_GroupKFold_summary_training113_protein.csv",
    index=False
)

for model_name, pred_table in oof_prediction_tables.items():
    pred_table.to_csv(
        OUT_DIR / f"internal_oof_predictions_{model_name}_training113_protein.csv",
        index=False
    )

best_model_name = internal_summary.iloc[0]["model"]
print("\nBest internal model by R²:", best_model_name)


# ============================================================
# 10. Train final fixed model on mixed-source training 113
# ============================================================

final_model = clone(models[best_model_name])
final_model.fit(X_train_all, y_train_all)

print("\nFinal protein model trained on mixed-source 113 samples.")
print("Best model:", best_model_name)
print("Training samples:", df_train_patches["global_sample_id"].nunique())
print("Training patches:", df_train_patches.shape[0])


# ============================================================
# 11. Fixed validation on reserved 36 samples
# ============================================================

y_patch_pred_val = final_model.predict(X_val)
y_patch_pred_val = np.asarray(y_patch_pred_val).ravel()

validation_pred = aggregate_patch_predictions_to_sample(
    patch_df=df_val_patches[["global_sample_id", "source", "sample_id", "treatment"]].copy(),
    y_patch_pred=y_patch_pred_val,
)

validation_pred = validation_pred.merge(
    quality_val[["global_sample_id", TARGET_COL]],
    on="global_sample_id",
    how="left"
)

validation_pred = validation_pred.rename(columns={TARGET_COL: "y_true"})

validation_pred["residual"] = validation_pred["y_pred"] - validation_pred["y_true"]
validation_pred["abs_error"] = validation_pred["residual"].abs()

fixed_validation_metrics = regression_metrics(
    y_true=validation_pred["y_true"].values,
    y_pred=validation_pred["y_pred"].values,
)

fixed_validation_metrics["model"] = best_model_name
fixed_validation_metrics["scenario"] = "fixed_mixed_source_train113_validation36_protein"

fixed_validation_metrics_df = pd.DataFrame([fixed_validation_metrics])

print("\n" + "=" * 90)
print("Fixed validation performance: mixed-source training 113 -> reserved validation 36")
print("=" * 90)
display(fixed_validation_metrics_df)

print("\nFixed validation sample-level predictions:")
display(validation_pred.sort_values(["source", "sample_id"]))

fixed_validation_metrics_df.to_csv(
    OUT_DIR / "validation36_metrics_fixed_protein.csv",
    index=False
)

validation_pred.to_csv(
    OUT_DIR / "validation36_predictions_fixed_protein.csv",
    index=False
)


# ============================================================
# 12. Fixed validation plots
# ============================================================

fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(validation_pred["y_true"], validation_pred["y_pred"], alpha=0.8)

min_v = min(validation_pred["y_true"].min(), validation_pred["y_pred"].min())
max_v = max(validation_pred["y_true"].max(), validation_pred["y_pred"].max())

ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")

ax.set_xlabel("Measured protein content")
ax.set_ylabel("Predicted protein content")
ax.set_title(
    f"Fixed validation protein prediction\n"
    f"{best_model_name}: R²={fixed_validation_metrics['r2']:.3f}, "
    f"RMSE={fixed_validation_metrics['rmse']:.3f}"
)

plt.tight_layout()

scatter_path = OUT_DIR / "validation36_scatter_fixed_protein.png"
plt.savefig(scatter_path, dpi=300)
plt.show()

print("Saved fixed validation scatter plot:", scatter_path)


fig, ax = plt.subplots(figsize=(6, 4))

ax.axhline(0, linestyle="--")
ax.scatter(validation_pred["y_true"], validation_pred["residual"], alpha=0.8)

ax.set_xlabel("Measured protein content")
ax.set_ylabel("Prediction residual")
ax.set_title("Fixed validation residual plot")

plt.tight_layout()

residual_path = OUT_DIR / "validation36_residual_fixed_protein.png"
plt.savefig(residual_path, dpi=300)
plt.show()

print("Saved fixed validation residual plot:", residual_path)


# ============================================================
# 13. Metrics by source and treatment for fixed validation
# ============================================================

source_rows = []

for source, sub in validation_pred.groupby("source"):
    m = regression_metrics(sub["y_true"].values, sub["y_pred"].values)
    m["source"] = source
    source_rows.append(m)

source_metrics_df = pd.DataFrame(source_rows)

print("\nFixed validation metrics by source:")
display(source_metrics_df)

source_metrics_df.to_csv(
    OUT_DIR / "validation36_metrics_by_source_fixed_protein.csv",
    index=False
)


treatment_rows = []

for (source, treatment), sub in validation_pred.groupby(["source", "treatment"]):
    if sub.shape[0] >= 2:
        m = regression_metrics(sub["y_true"].values, sub["y_pred"].values)
    else:
        m = {
            "n_samples": sub.shape[0],
            "r2": np.nan,
            "rmse": np.nan,
            "mae": np.nan,
            "bias": np.nan,
            "rpd": np.nan,
        }

    m["source"] = source
    m["treatment"] = treatment
    treatment_rows.append(m)

treatment_metrics_df = pd.DataFrame(treatment_rows)

print("\nFixed validation metrics by source/treatment:")
display(treatment_metrics_df)

treatment_metrics_df.to_csv(
    OUT_DIR / "validation36_metrics_by_source_treatment_fixed_protein.csv",
    index=False
)


# ============================================================
# 14. Save fixed model package
# ============================================================

model_package = {
    "analysis_type": "two-source mixed protein regression model",
    "description": "Training set = 89 samples from Source A + 24 samples from Source B. Validation set = 24 Source A + 12 Source B.",
    "random_state": RANDOM_STATE,
    "target_col": TARGET_COL,
    "wl_cols": wl_cols,
    "wavelengths": wavelengths,
    "best_model_name": best_model_name,
    "final_model": final_model,
    "models_available": list(models.keys()),
    "train_ids": train_ids,
    "validation_ids_reserved": val_ids,
    "source_a_patch_csv": str(A_PATCH_CSV),
    "source_b_patch_csv": str(B_PATCH_CSV),
    "source_b_quality_xlsx": str(B_QUALITY_XLSX),
    "fixed_validation_metrics": fixed_validation_metrics,
}

joblib.dump(
    model_package,
    OUT_DIR / "mixed_source_training113_protein_model_package.joblib"
)

quality_all.to_csv(
    OUT_DIR / "combined_A_B_protein_quality_all149.csv",
    index=False
)

df_all_spectra[
    ["global_sample_id", "source", "sample_id", "treatment"]
].drop_duplicates().to_csv(
    OUT_DIR / "combined_A_B_sample_index_all149.csv",
    index=False
)

print("\nSaved fixed protein model package:")
print(OUT_DIR / "mixed_source_training113_protein_model_package.joblib")


# ============================================================
# 15. Optional repeated two-source holdout protein regression
# ============================================================

# The fixed 113/36 analysis above always runs.
# Set this to True only when the full repeated validation is required.
RUN_REPEATED_VALIDATION = False
N_REPEATS = 50

MODEL_NAMES_TO_RUN = [
    "PLSR_10",
    "PLSR_15",
    "SVR_RBF",
    "RandomForest",
    "ExtraTrees",
    "Ridge",
    "ElasticNet",
]

if RUN_REPEATED_VALIDATION:
    print("\n" + "=" * 90)
    print("Start repeated two-source holdout protein regression")
    print("=" * 90)
    print("Repeated validation output folder:", REPEAT_OUT_DIR)
    print("Number of repeats:", N_REPEATS)
    print("Models:", MODEL_NAMES_TO_RUN)


    def run_one_repeat_one_model_protein(repeat_id, model_name, model_template):
        train_ids_repeat, val_ids_repeat = create_two_source_split_for_repeat(
            repeat_id=repeat_id,
            random_state=RANDOM_STATE
        )

        quality_train_repeat = quality_all[
            quality_all["global_sample_id"].isin(train_ids_repeat)
        ].copy()

        quality_val_repeat = quality_all[
            quality_all["global_sample_id"].isin(val_ids_repeat)
        ].copy()

        target_map_train = quality_train_repeat.set_index("global_sample_id")[TARGET_COL].to_dict()
        target_map_val = quality_val_repeat.set_index("global_sample_id")[TARGET_COL].to_dict()

        df_train_patches_repeat = df_all_spectra[
            df_all_spectra["global_sample_id"].isin(train_ids_repeat)
        ].copy()

        df_val_patches_repeat = df_all_spectra[
            df_all_spectra["global_sample_id"].isin(val_ids_repeat)
        ].copy()

        df_train_patches_repeat["y"] = (
            df_train_patches_repeat["global_sample_id"]
            .map(target_map_train)
            .astype(float)
        )

        df_val_patches_repeat["y_true"] = (
            df_val_patches_repeat["global_sample_id"]
            .map(target_map_val)
            .astype(float)
        )

        X_train_repeat = df_train_patches_repeat[wl_cols].values.astype(float)
        y_train_repeat = df_train_patches_repeat["y"].values.astype(float)

        X_val_repeat = df_val_patches_repeat[wl_cols].values.astype(float)

        model = clone(model_template)
        model.fit(X_train_repeat, y_train_repeat)

        y_patch_pred_val = model.predict(X_val_repeat)
        y_patch_pred_val = np.asarray(y_patch_pred_val).ravel()

        validation_pred_repeat = aggregate_patch_predictions_to_sample(
            patch_df=df_val_patches_repeat[
                ["global_sample_id", "source", "sample_id", "treatment"]
            ].copy(),
            y_patch_pred=y_patch_pred_val,
        )

        validation_pred_repeat = validation_pred_repeat.merge(
            quality_val_repeat[["global_sample_id", TARGET_COL]],
            on="global_sample_id",
            how="left"
        )

        validation_pred_repeat = validation_pred_repeat.rename(columns={TARGET_COL: "y_true"})

        validation_pred_repeat["residual"] = (
            validation_pred_repeat["y_pred"] - validation_pred_repeat["y_true"]
        )

        validation_pred_repeat["abs_error"] = validation_pred_repeat["residual"].abs()
        validation_pred_repeat["repeat"] = repeat_id
        validation_pred_repeat["model"] = model_name

        metrics = regression_metrics(
            y_true=validation_pred_repeat["y_true"].values,
            y_pred=validation_pred_repeat["y_pred"].values,
        )

        metrics.update({
            "repeat": repeat_id,
            "model": model_name,
            "n_train_samples": len(train_ids_repeat),
            "n_validation_samples": len(val_ids_repeat),
            "train_A_n": sum([x.startswith("A_") for x in train_ids_repeat]),
            "train_B_n": sum([x.startswith("B_") for x in train_ids_repeat]),
            "validation_A_n": sum([x.startswith("A_") for x in val_ids_repeat]),
            "validation_B_n": sum([x.startswith("B_") for x in val_ids_repeat]),
            "train_protein_min": quality_train_repeat[TARGET_COL].min(),
            "train_protein_mean": quality_train_repeat[TARGET_COL].mean(),
            "train_protein_max": quality_train_repeat[TARGET_COL].max(),
            "validation_protein_min": quality_val_repeat[TARGET_COL].min(),
            "validation_protein_mean": quality_val_repeat[TARGET_COL].mean(),
            "validation_protein_max": quality_val_repeat[TARGET_COL].max(),
        })

        return metrics, validation_pred_repeat, quality_train_repeat, quality_val_repeat


    all_metrics = []
    all_predictions = []
    all_validation_quality = []

    for model_name in MODEL_NAMES_TO_RUN:
        if model_name not in models:
            raise ValueError(
                f"{model_name} not found in models. Available models: {list(models.keys())}"
            )

        print("\n" + "=" * 90)
        print("Repeated protein regression model:", model_name)
        print("=" * 90)

        for repeat_id in range(N_REPEATS):
            metrics, validation_pred_repeat, quality_train_repeat, quality_val_repeat = run_one_repeat_one_model_protein(
                repeat_id=repeat_id,
                model_name=model_name,
                model_template=models[model_name],
            )

            all_metrics.append(metrics)
            all_predictions.append(validation_pred_repeat)

            temp_val_quality = quality_val_repeat.copy()
            temp_val_quality["repeat"] = repeat_id
            temp_val_quality["model"] = model_name
            all_validation_quality.append(temp_val_quality)

            if (repeat_id + 1) % 10 == 0:
                print(f"Finished {repeat_id + 1}/{N_REPEATS} repeats for {model_name}")


    metrics_df = pd.DataFrame(all_metrics)
    predictions_df = pd.concat(all_predictions, axis=0, ignore_index=True)
    validation_quality_df = pd.concat(all_validation_quality, axis=0, ignore_index=True)

    metrics_path = REPEAT_OUT_DIR / "repeated_validation_metrics_each_repeat_protein.csv"
    predictions_path = REPEAT_OUT_DIR / "repeated_validation_sample_predictions_protein.csv"
    quality_path = REPEAT_OUT_DIR / "repeated_validation_quality_table_protein.csv"

    metrics_df.to_csv(metrics_path, index=False)
    predictions_df.to_csv(predictions_path, index=False)
    validation_quality_df.to_csv(quality_path, index=False)

    print("\nSaved raw repeated-validation outputs:")
    print(metrics_path)
    print(predictions_path)
    print(quality_path)


    # ============================================================
    # 16. Repeated validation summary by model
    # ============================================================

    summary_by_model = (
        metrics_df
        .groupby("model")[
            [
                "r2",
                "rmse",
                "mae",
                "bias",
                "rpd",
                "validation_protein_min",
                "validation_protein_mean",
                "validation_protein_max",
            ]
        ]
        .agg(["mean", "std", "min", "max"])
    )

    summary_by_model.columns = [
        f"{metric}_{stat}"
        for metric, stat in summary_by_model.columns
    ]

    summary_by_model = summary_by_model.reset_index()

    summary_by_model = summary_by_model.sort_values(
        ["r2_mean", "rpd_mean"],
        ascending=False
    ).reset_index(drop=True)

    summary_path = REPEAT_OUT_DIR / "repeated_validation_summary_by_model_protein.csv"
    summary_by_model.to_csv(summary_path, index=False)

    print("\nRepeated protein regression summary by model:")
    display(summary_by_model)

    print("\nFormatted summary:")
    for _, row in summary_by_model.iterrows():
        print(
            f"{row['model']}: "
            f"R²={row['r2_mean']:.3f} ± {row['r2_std']:.3f}; "
            f"RMSE={row['rmse_mean']:.3f} ± {row['rmse_std']:.3f}; "
            f"MAE={row['mae_mean']:.3f} ± {row['mae_std']:.3f}; "
            f"RPD={row['rpd_mean']:.3f} ± {row['rpd_std']:.3f}; "
            f"Bias={row['bias_mean']:.3f} ± {row['bias_std']:.3f}"
        )

    print("\nSaved repeated validation model summary:")
    print(summary_path)


    # ============================================================
    # 17. Best repeated model plots and source-level metrics
    # ============================================================

    best_repeat_model = summary_by_model.iloc[0]["model"]

    print("\nBest repeated-validation model by mean R²:", best_repeat_model)

    best_predictions = predictions_df[
        predictions_df["model"] == best_repeat_model
    ].copy()

    agg_metrics = regression_metrics(
        y_true=best_predictions["y_true"].values,
        y_pred=best_predictions["y_pred"].values,
    )

    print("\nAggregated metrics for best repeated model:")
    print(agg_metrics)

    fig, ax = plt.subplots(figsize=(5, 5))

    ax.scatter(best_predictions["y_true"], best_predictions["y_pred"], alpha=0.4)

    min_v = min(best_predictions["y_true"].min(), best_predictions["y_pred"].min())
    max_v = max(best_predictions["y_true"].max(), best_predictions["y_pred"].max())

    ax.plot([min_v, max_v], [min_v, max_v], linestyle="--")

    ax.set_xlabel("Measured protein content")
    ax.set_ylabel("Predicted protein content")
    ax.set_title(
        f"Repeated validation protein prediction\n"
        f"Best model: {best_repeat_model}"
    )

    plt.tight_layout()

    agg_scatter_path = REPEAT_OUT_DIR / "best_model_aggregated_scatter_protein.png"
    plt.savefig(agg_scatter_path, dpi=300)
    plt.show()

    print("\nSaved aggregated scatter plot:")
    print(agg_scatter_path)


    fig, ax = plt.subplots(figsize=(6, 4))

    ax.axhline(0, linestyle="--")
    ax.scatter(best_predictions["y_true"], best_predictions["residual"], alpha=0.4)

    ax.set_xlabel("Measured protein content")
    ax.set_ylabel("Prediction residual")
    ax.set_title(
        f"Repeated validation residuals\n"
        f"Best model: {best_repeat_model}"
    )

    plt.tight_layout()

    agg_residual_path = REPEAT_OUT_DIR / "best_model_aggregated_residual_protein.png"
    plt.savefig(agg_residual_path, dpi=300)
    plt.show()

    print("\nSaved aggregated residual plot:")
    print(agg_residual_path)


    # Source-level metrics for best repeated model
    source_rows = []

    for (repeat_id, source), sub in best_predictions.groupby(["repeat", "source"]):
        m = regression_metrics(sub["y_true"].values, sub["y_pred"].values)
        m["repeat"] = repeat_id
        m["source"] = source
        source_rows.append(m)

    source_metrics_df = pd.DataFrame(source_rows)

    source_summary = (
        source_metrics_df
        .groupby("source")[["r2", "rmse", "mae", "bias", "rpd"]]
        .agg(["mean", "std", "min", "max"])
    )

    source_metrics_path = REPEAT_OUT_DIR / "best_model_metrics_by_source_each_repeat_protein.csv"
    source_summary_path = REPEAT_OUT_DIR / "best_model_metrics_by_source_summary_protein.csv"

    source_metrics_df.to_csv(source_metrics_path, index=False)
    source_summary.to_csv(source_summary_path)

    print("\nBest model source-level repeated metrics:")
    display(source_summary)

    print("\nSaved source-level metrics:")
    print(source_metrics_path)
    print(source_summary_path)


    # B treatment-level metrics for best repeated model
    treatment_rows = []

    best_predictions_b = best_predictions[
        best_predictions["source"] == "B_second"
    ].copy()

    for (repeat_id, treatment), sub in best_predictions_b.groupby(["repeat", "treatment"]):
        if sub.shape[0] >= 2:
            m = regression_metrics(sub["y_true"].values, sub["y_pred"].values)
        else:
            m = {
                "n_samples": sub.shape[0],
                "r2": np.nan,
                "rmse": np.nan,
                "mae": np.nan,
                "bias": np.nan,
                "rpd": np.nan,
            }

        m["repeat"] = repeat_id
        m["treatment"] = treatment
        treatment_rows.append(m)

    treatment_metrics_df = pd.DataFrame(treatment_rows)

    treatment_summary = (
        treatment_metrics_df
        .groupby("treatment")[["r2", "rmse", "mae", "bias", "rpd"]]
        .agg(["mean", "std", "min", "max"])
    )

    treatment_metrics_path = REPEAT_OUT_DIR / "best_model_B_treatment_metrics_each_repeat_protein.csv"
    treatment_summary_path = REPEAT_OUT_DIR / "best_model_B_treatment_metrics_summary_protein.csv"

    treatment_metrics_df.to_csv(treatment_metrics_path, index=False)
    treatment_summary.to_csv(treatment_summary_path)

    print("\nBest model B-treatment repeated metrics:")
    display(treatment_summary)

    print("\nSaved B-treatment metrics:")
    print(treatment_metrics_path)
    print(treatment_summary_path)


    # ============================================================
    # 18. Metric distribution boxplot
    # ============================================================

    plot_model_df = metrics_df[
        metrics_df["model"] == best_repeat_model
    ].copy()

    fig, ax = plt.subplots(figsize=(7, 5))

    metric_names = ["r2", "rmse", "mae", "rpd"]
    data_to_plot = [plot_model_df[m].values for m in metric_names]

    ax.boxplot(data_to_plot, labels=metric_names)

    ax.set_ylabel("Score")
    ax.set_title(
        "Repeated two-source holdout protein regression\n"
        f"Best model: {best_repeat_model}, n={N_REPEATS}"
    )

    plt.xticks(rotation=20, ha="right")
    plt.tight_layout()

    boxplot_path = REPEAT_OUT_DIR / "best_model_metric_distribution_boxplot_protein.png"
    plt.savefig(boxplot_path, dpi=300)
    plt.show()

    print("\nSaved metric distribution plot:")
    print(boxplot_path)


    # ============================================================
    # 19. Save text summary
    # ============================================================

    best_row = summary_by_model.iloc[0]

    summary_txt = f"""
    Repeated two-source holdout validation summary - Protein regression

    Design:
    - Total sample pool: Source A 113 + Source B 36 = 149 samples
    - Each repeat training set: 89 Source A + 24 Source B = 113 samples
    - Each repeat validation set: 24 Source A + 12 Source B = 36 samples
    - Source B validation set was treatment-balanced:
      4 Control + 4 150% N + 4 50% water

    Target:
    {TARGET_COL}

    Method:
    - Patch-level spectra were used for model training.
    - Patch-level predictions were averaged to sample-level predictions.
    - Spectral regressors were trained only on the training 113 samples in each repeat.
    - Validation performance was evaluated on the reserved 36 samples.

    Number of repeats:
    {N_REPEATS}

    Best model:
    {best_repeat_model}

    Best model repeated-validation performance:
    - R²: {best_row['r2_mean']:.3f} ± {best_row['r2_std']:.3f}
    - RMSE: {best_row['rmse_mean']:.3f} ± {best_row['rmse_std']:.3f}
    - MAE: {best_row['mae_mean']:.3f} ± {best_row['mae_std']:.3f}
    - RPD: {best_row['rpd_mean']:.3f} ± {best_row['rpd_std']:.3f}
    - Bias: {best_row['bias_mean']:.3f} ± {best_row['bias_std']:.3f}

    Important note:
    This is a mixed-source repeated validation, not the strict A113 -> B36 external validation.
    """

    summary_txt_path = REPEAT_OUT_DIR / "repeated_validation_summary_protein.txt"

    with open(summary_txt_path, "w", encoding="utf-8") as f:
        f.write(summary_txt)

    print("\nSaved text summary:")
    print(summary_txt_path)

    print("\nProtein regression analysis finished.")
    print("Fixed model outputs saved to:")
    print(OUT_DIR)
    print("Repeated validation outputs saved to:")
    print(REPEAT_OUT_DIR)

else:
    print(
        "\nRepeated two-source holdout validation was skipped. "
        "Set RUN_REPEATED_VALIDATION = True to run the full 50-repeat analysis."
    )


In [ ]:
# ============================================================
# Generate final protein regression plots and Top 10 VIP wavelengths
# Run this AFTER the protein regression code
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ------------------------------------------------------------
# 0. Figure style
# ------------------------------------------------------------

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["xtick.labelsize"] = 20
plt.rcParams["ytick.labelsize"] = 20
plt.rcParams["legend.fontsize"] = 20

PLOT_OUT_DIR = OUT_DIR / "Final_protein_plots_and_VIP"
PLOT_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", PLOT_OUT_DIR)


# ============================================================
# 1. Helper functions
# ============================================================

def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)

    if rmse == 0:
        rpd = np.inf
    else:
        rpd = np.std(y_true, ddof=1) / rmse

    return {
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "rpd": rpd,
    }


def plot_measured_vs_predicted(df, title, output_path):
    y_true = df["y_true"].values.astype(float)
    y_pred = df["y_pred"].values.astype(float)

    metrics = calc_regression_metrics(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(y_true, y_pred, alpha=0.8, s=70)

    min_v = min(y_true.min(), y_pred.min())
    max_v = max(y_true.max(), y_pred.max())

    padding = (max_v - min_v) * 0.08
    min_v = min_v - padding
    max_v = max_v + padding

    ax.plot([min_v, max_v], [min_v, max_v], linestyle="--", linewidth=2)

    # Optional fitted regression line
    slope, intercept = np.polyfit(y_true, y_pred, 1)
    x_line = np.linspace(min_v, max_v, 100)
    y_line = slope * x_line + intercept
    ax.plot(x_line, y_line, linewidth=2)

    ax.set_xlim(min_v, max_v)
    ax.set_ylim(min_v, max_v)

    ax.set_xlabel("Measured protein content (%)")
    ax.set_ylabel("Predicted protein content (%)")

    ax.set_title(title)

    text = (
        f"R² = {metrics['r2']:.3f}\n"
        f"RMSE = {metrics['rmse']:.3f}\n"
        f"RPD = {metrics['rpd']:.3f}"
    )

    ax.text(
        0.05,
        0.95,
        text,
        transform=ax.transAxes,
        verticalalignment="top",
        fontsize=20,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.8)
    )

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return metrics


def plot_residuals(df, title, output_path):
    plot_df = df.copy()
    plot_df["residual"] = plot_df["y_pred"] - plot_df["y_true"]

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.axhline(0, linestyle="--", linewidth=2)
    ax.scatter(
        plot_df["y_true"],
        plot_df["residual"],
        alpha=0.8,
        s=70
    )

    ax.set_xlabel("Measured protein content (%)")
    ax.set_ylabel("Prediction residual (%)")
    ax.set_title(title)

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def calculate_vip_from_plsr_pipeline(plsr_pipeline, wavelengths):
    """
    Calculate VIP scores from a fitted sklearn Pipeline containing PLSRegression
    as the step named 'reg'.

    The model should be fitted before running this function.
    """

    if "reg" not in plsr_pipeline.named_steps:
        raise ValueError("The pipeline does not contain a step named 'reg'.")

    pls = plsr_pipeline.named_steps["reg"]

    if not hasattr(pls, "x_scores_"):
        raise ValueError("The regression model does not have PLS attributes. It may not be a fitted PLSR model.")

    T = pls.x_scores_          # score matrix, shape: n_samples x n_components
    W = pls.x_weights_         # weight matrix, shape: n_features x n_components
    Q = pls.y_loadings_        # y loadings, shape: n_targets x n_components

    p, h = W.shape

    # Sum of squares explained in Y by each component
    ssy = np.sum(T ** 2, axis=0) * np.sum(Q ** 2, axis=0)

    total_ssy = np.sum(ssy)

    if total_ssy == 0:
        raise ValueError("Total explained Y variance is zero; VIP cannot be calculated.")

    # Normalize weights by component
    W_norm = W / np.sqrt(np.sum(W ** 2, axis=0, keepdims=True))

    vip = np.sqrt(
        p * np.sum((W_norm ** 2) * ssy.reshape(1, -1), axis=1) / total_ssy
    )

    vip_df = pd.DataFrame({
        "wavelength": wavelengths,
        "VIP": vip,
    })

    vip_df = vip_df.sort_values("VIP", ascending=False).reset_index(drop=True)

    return vip_df


def plot_top_vip(vip_df, top_n, output_path):
    top_df = vip_df.head(top_n).copy()
    top_df = top_df.sort_values("VIP", ascending=True)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.barh(
        top_df["wavelength"].astype(str),
        top_df["VIP"]
    )

    ax.axvline(1.0, linestyle="--", linewidth=2)

    ax.set_xlabel("VIP score")
    ax.set_ylabel("Wavelength (nm)")
    ax.set_title(f"Top {top_n} VIP wavelengths")

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# 2. Fixed validation regression plot for final best model
# ============================================================

fixed_plot_df = validation_pred.copy()

fixed_plot_path = PLOT_OUT_DIR / "fixed_validation_measured_vs_predicted_protein.png"
fixed_residual_path = PLOT_OUT_DIR / "fixed_validation_residual_protein.png"

fixed_metrics = plot_measured_vs_predicted(
    df=fixed_plot_df,
    title=f"Protein prediction - fixed validation\nBest model: {best_model_name}",
    output_path=fixed_plot_path,
)

plot_residuals(
    df=fixed_plot_df,
    title=f"Protein residuals - fixed validation\nBest model: {best_model_name}",
    output_path=fixed_residual_path,
)

fixed_metrics_df = pd.DataFrame([fixed_metrics])
fixed_metrics_df["model"] = best_model_name
fixed_metrics_df["dataset"] = "fixed_validation"

fixed_metrics_df.to_csv(
    PLOT_OUT_DIR / "fixed_validation_metrics_for_plot.csv",
    index=False
)

print("\nFixed validation plot saved:")
print(fixed_plot_path)
print(fixed_residual_path)
display(fixed_metrics_df)


# ============================================================
# 3. Repeated validation aggregated plot for best repeated model
#    This section runs only if repeated-validation variables exist.
# ============================================================

if "predictions_df" in globals() and "summary_by_model" in globals():
    best_repeat_model = summary_by_model.sort_values(
        ["r2_mean", "rpd_mean"],
        ascending=False
    ).iloc[0]["model"]

    repeated_plot_df = predictions_df[
        predictions_df["model"] == best_repeat_model
    ].copy()

    repeated_plot_path = PLOT_OUT_DIR / "repeated_validation_aggregated_measured_vs_predicted_protein.png"
    repeated_residual_path = PLOT_OUT_DIR / "repeated_validation_aggregated_residual_protein.png"

    repeated_metrics = plot_measured_vs_predicted(
        df=repeated_plot_df,
        title=f"Protein prediction - repeated validation\nBest model: {best_repeat_model}",
        output_path=repeated_plot_path,
    )

    plot_residuals(
        df=repeated_plot_df,
        title=f"Protein residuals - repeated validation\nBest model: {best_repeat_model}",
        output_path=repeated_residual_path,
    )

    repeated_metrics_df = pd.DataFrame([repeated_metrics])
    repeated_metrics_df["model"] = best_repeat_model
    repeated_metrics_df["dataset"] = "repeated_validation_aggregated"

    repeated_metrics_df.to_csv(
        PLOT_OUT_DIR / "repeated_validation_aggregated_metrics_for_plot.csv",
        index=False
    )

    print("\nRepeated validation aggregated plot saved:")
    print(repeated_plot_path)
    print(repeated_residual_path)
    display(repeated_metrics_df)

else:
    print("\nRepeated-validation variables were not found. Skipped repeated-validation aggregated plot.")


# ============================================================
# 4. VIP calculation for PLSR best model
# ============================================================

# VIP is only meaningful for PLSR models.
# In your current result, the best fixed model is PLSR_15.

if "PLSR" in best_model_name.upper():

    vip_df = calculate_vip_from_plsr_pipeline(
        plsr_pipeline=final_model,
        wavelengths=wavelengths,
    )

    vip_path = PLOT_OUT_DIR / "VIP_scores_best_PLSR_model.csv"
    top10_vip_path = PLOT_OUT_DIR / "Top10_VIP_wavelengths_best_PLSR_model.csv"
    vip_plot_path = PLOT_OUT_DIR / "Top10_VIP_wavelengths_best_PLSR_model.png"

    vip_df.to_csv(vip_path, index=False)

    top10_vip_df = vip_df.head(10).copy()
    top10_vip_df.to_csv(top10_vip_path, index=False)

    print("\nTop 10 VIP wavelengths:")
    display(top10_vip_df)

    plot_top_vip(
        vip_df=vip_df,
        top_n=10,
        output_path=vip_plot_path,
    )

    print("\nVIP files saved:")
    print(vip_path)
    print(top10_vip_path)
    print(vip_plot_path)

else:
    print(
        f"\nBest fixed model is {best_model_name}, not PLSR. "
        "VIP is only calculated for PLSR models."
    )


# ============================================================
# 5. Optional: Find samples with largest fixed-validation errors
# ============================================================

largest_error_df = fixed_plot_df.copy()
largest_error_df["residual"] = largest_error_df["y_pred"] - largest_error_df["y_true"]
largest_error_df["abs_error"] = largest_error_df["residual"].abs()

largest_error_df = largest_error_df.sort_values(
    "abs_error",
    ascending=False
)

largest_error_path = PLOT_OUT_DIR / "Top10_largest_prediction_errors_fixed_validation.csv"
largest_error_df.head(10).to_csv(largest_error_path, index=False)

print("\nTop 10 largest fixed-validation prediction errors:")
display(largest_error_df.head(10))

print("\nSaved largest-error table:")
print(largest_error_path)

print("\nAll protein plots and VIP outputs saved to:")
print(PLOT_OUT_DIR)

In [ ]:
# ============================================================
# Final protein regression plots + VIP
#
# Style:
# - Arial
# - Font size 20
# - No title
# - No fitted regression line
# - No textbox around metrics
#
# Run this AFTER the protein regression model code.
#
# Required variables:
# - oof_prediction_tables
# - best_model_name
# - validation_pred
# - final_model
# - wavelengths
# - OUT_DIR
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================
# 0. Figure style
# ============================================================

plt.rcParams["font.family"] = "Arial"
plt.rcParams["font.size"] = 20
plt.rcParams["axes.labelsize"] = 20
plt.rcParams["axes.titlesize"] = 20
plt.rcParams["xtick.labelsize"] = 20
plt.rcParams["ytick.labelsize"] = 20
plt.rcParams["legend.fontsize"] = 20

FINAL_PLOT_DIR = OUT_DIR / "Final_protein_figures_clean"
FINAL_PLOT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", FINAL_PLOT_DIR)


# ============================================================
# 1. Check required variables
# ============================================================

required_vars = [
    "oof_prediction_tables",
    "best_model_name",
    "validation_pred",
    "final_model",
    "wavelengths",
    "OUT_DIR",
]

for v in required_vars:
    if v not in globals():
        raise RuntimeError(f"Missing required variable: {v}")

if best_model_name not in oof_prediction_tables:
    raise RuntimeError(
        f"{best_model_name} not found in oof_prediction_tables. "
        f"Available models: {list(oof_prediction_tables.keys())}"
    )

print("Best model:", best_model_name)


# ============================================================
# 2. Helper functions
# ============================================================

def calc_regression_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    r2 = r2_score(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    bias = np.mean(y_pred - y_true)

    if rmse == 0:
        rpd = np.inf
    else:
        rpd = np.std(y_true, ddof=1) / rmse

    return {
        "r2": r2,
        "rmse": rmse,
        "mae": mae,
        "bias": bias,
        "rpd": rpd,
    }


def prepare_prediction_df(df):
    plot_df = df.copy()

    if "y_true" not in plot_df.columns or "y_pred" not in plot_df.columns:
        raise ValueError("DataFrame must contain y_true and y_pred columns.")

    plot_df["y_true"] = plot_df["y_true"].astype(float)
    plot_df["y_pred"] = plot_df["y_pred"].astype(float)
    plot_df["residual"] = plot_df["y_pred"] - plot_df["y_true"]
    plot_df["abs_error"] = plot_df["residual"].abs()

    return plot_df


def plot_regression_clean(df, output_path):
    plot_df = prepare_prediction_df(df)

    y_true = plot_df["y_true"].values
    y_pred = plot_df["y_pred"].values

    metrics = calc_regression_metrics(y_true, y_pred)

    fig, ax = plt.subplots(figsize=(7, 7))

    ax.scatter(
        y_true,
        y_pred,
        s=80,
        alpha=0.85
    )

    min_v = min(y_true.min(), y_pred.min())
    max_v = max(y_true.max(), y_pred.max())
    padding = (max_v - min_v) * 0.08

    min_v -= padding
    max_v += padding

    # 1:1 line only
    ax.plot(
        [min_v, max_v],
        [min_v, max_v],
        linestyle="--",
        linewidth=2,
        color="black"
    )

    ax.set_xlim(min_v, max_v)
    ax.set_ylim(min_v, max_v)

    ax.set_xlabel("Measured protein content (%)")
    ax.set_ylabel("Predicted protein content (%)")

    # No title
    # No textbox
    ax.text(
        0.05,
        0.95,
        f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.3f}\nRPD = {metrics['rpd']:.3f}",
        transform=ax.transAxes,
        va="top",
        fontsize=20
    )

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()

    return metrics, plot_df


def plot_residual_clean(df, output_path):
    plot_df = prepare_prediction_df(df)

    fig, ax = plt.subplots(figsize=(8, 6))

    ax.axhline(
        0,
        linestyle="--",
        linewidth=2,
        color="black"
    )

    ax.scatter(
        plot_df["y_true"],
        plot_df["residual"],
        s=80,
        alpha=0.85
    )

    ax.set_xlabel("Measured protein content (%)")
    ax.set_ylabel("Prediction residual (%)")

    # No title

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


def calculate_vip_from_plsr_pipeline(plsr_pipeline, wavelengths):
    """
    Calculate VIP scores from a fitted sklearn Pipeline containing
    PLSRegression as the step named 'reg'.
    """

    if "reg" not in plsr_pipeline.named_steps:
        raise ValueError("The pipeline does not contain a step named 'reg'.")

    pls = plsr_pipeline.named_steps["reg"]

    if not hasattr(pls, "x_scores_"):
        raise ValueError(
            "The regression model does not have PLS attributes. "
            "It may not be a fitted PLSRegression model."
        )

    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_

    p, h = W.shape

    ssy = np.sum(T ** 2, axis=0) * np.sum(Q ** 2, axis=0)
    total_ssy = np.sum(ssy)

    if total_ssy == 0:
        raise ValueError("Total explained Y variance is zero; VIP cannot be calculated.")

    W_norm = W / np.sqrt(np.sum(W ** 2, axis=0, keepdims=True))

    vip = np.sqrt(
        p * np.sum((W_norm ** 2) * ssy.reshape(1, -1), axis=1) / total_ssy
    )

    vip_df = pd.DataFrame({
        "wavelength": np.asarray(wavelengths, dtype=float),
        "VIP": vip,
    })

    vip_df = vip_df.sort_values("VIP", ascending=False).reset_index(drop=True)

    return vip_df


def plot_top_vip_clean(vip_df, top_n, output_path):
    top_df = vip_df.head(top_n).copy()
    top_df = top_df.sort_values("VIP", ascending=True)
    top_df["wavelength_label"] = top_df["wavelength"].round(2).astype(str)

    fig, ax = plt.subplots(figsize=(9, 7))

    ax.barh(
        top_df["wavelength_label"],
        top_df["VIP"]
    )

    ax.axvline(
        1.0,
        linestyle="--",
        linewidth=2,
        color="black"
    )

    ax.set_xlabel("VIP score")
    ax.set_ylabel("Wavelength (nm)")

    # No title

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches="tight")
    plt.show()


# ============================================================
# 3. Original model performance: GroupKFold OOF predictions
# ============================================================

original_oof_df = oof_prediction_tables[best_model_name].copy()
original_oof_df = prepare_prediction_df(original_oof_df)

original_plot_path = FINAL_PLOT_DIR / "original_model_GroupKFold_measured_vs_predicted_protein_clean.png"
original_residual_path = FINAL_PLOT_DIR / "original_model_GroupKFold_residual_protein_clean.png"

original_metrics, original_oof_df = plot_regression_clean(
    df=original_oof_df,
    output_path=original_plot_path
)

plot_residual_clean(
    df=original_oof_df,
    output_path=original_residual_path
)

original_metrics_df = pd.DataFrame([original_metrics])
original_metrics_df["model"] = best_model_name
original_metrics_df["dataset"] = "Original model - GroupKFold OOF"

original_oof_df.to_csv(
    FINAL_PLOT_DIR / "original_model_GroupKFold_predictions_clean.csv",
    index=False
)

original_metrics_df.to_csv(
    FINAL_PLOT_DIR / "original_model_GroupKFold_metrics_clean.csv",
    index=False
)

print("\nOriginal model performance:")
display(original_metrics_df)


# ============================================================
# 4. Reserved validation performance
# ============================================================

validation_plot_df = validation_pred.copy()
validation_plot_df = prepare_prediction_df(validation_plot_df)

validation_plot_path = FINAL_PLOT_DIR / "validation36_measured_vs_predicted_protein_clean.png"
validation_residual_path = FINAL_PLOT_DIR / "validation36_residual_protein_clean.png"

validation_metrics, validation_plot_df = plot_regression_clean(
    df=validation_plot_df,
    output_path=validation_plot_path
)

plot_residual_clean(
    df=validation_plot_df,
    output_path=validation_residual_path
)

validation_metrics_df = pd.DataFrame([validation_metrics])
validation_metrics_df["model"] = best_model_name
validation_metrics_df["dataset"] = "Reserved validation 36"

validation_plot_df.to_csv(
    FINAL_PLOT_DIR / "validation36_predictions_clean.csv",
    index=False
)

validation_metrics_df.to_csv(
    FINAL_PLOT_DIR / "validation36_metrics_clean.csv",
    index=False
)

print("\nReserved validation performance:")
display(validation_metrics_df)


# ============================================================
# 5. Original vs validation performance comparison table
# ============================================================

performance_compare = pd.concat(
    [original_metrics_df, validation_metrics_df],
    axis=0,
    ignore_index=True
)

performance_compare = performance_compare[
    ["dataset", "model", "r2", "rmse", "mae", "bias", "rpd"]
]

performance_compare.to_csv(
    FINAL_PLOT_DIR / "original_vs_validation_performance_comparison_clean.csv",
    index=False
)

print("\nOriginal model vs reserved validation performance:")
display(performance_compare)


# ============================================================
# 6. Combined figure: original vs validation
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 7))

for ax, df in zip(
    axes,
    [original_oof_df, validation_plot_df]
):
    y_true = df["y_true"].astype(float).values
    y_pred = df["y_pred"].astype(float).values

    metrics = calc_regression_metrics(y_true, y_pred)

    ax.scatter(
        y_true,
        y_pred,
        s=70,
        alpha=0.85
    )

    min_v = min(y_true.min(), y_pred.min())
    max_v = max(y_true.max(), y_pred.max())
    padding = (max_v - min_v) * 0.08

    min_v -= padding
    max_v += padding

    # 1:1 line only
    ax.plot(
        [min_v, max_v],
        [min_v, max_v],
        linestyle="--",
        linewidth=2,
        color="black"
    )

    ax.set_xlim(min_v, max_v)
    ax.set_ylim(min_v, max_v)

    ax.set_xlabel("Measured protein content (%)")
    ax.set_ylabel("Predicted protein content (%)")

    # No title
    # No textbox
    ax.text(
        0.05,
        0.95,
        f"R² = {metrics['r2']:.3f}\nRMSE = {metrics['rmse']:.3f}\nRPD = {metrics['rpd']:.3f}",
        transform=ax.transAxes,
        va="top",
        fontsize=20
    )

    ax.tick_params(direction="out", length=6, width=1.5)

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)

plt.tight_layout()

combined_plot_path = FINAL_PLOT_DIR / "original_vs_validation_regression_plots_clean.png"
plt.savefig(combined_plot_path, dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved combined plot:")
print(combined_plot_path)


# ============================================================
# 7. VIP scores and Top 10 VIP wavelengths
# ============================================================

if "PLSR" in str(best_model_name).upper():

    vip_df = calculate_vip_from_plsr_pipeline(
        plsr_pipeline=final_model,
        wavelengths=wavelengths
    )

    vip_path = FINAL_PLOT_DIR / "VIP_scores_best_PLSR_model.csv"
    top10_vip_path = FINAL_PLOT_DIR / "Top10_VIP_wavelengths_best_PLSR_model.csv"
    vip_plot_path = FINAL_PLOT_DIR / "Top10_VIP_wavelengths_best_PLSR_model_clean.png"

    vip_df.to_csv(vip_path, index=False)

    top10_vip_df = vip_df.head(10).copy()
    top10_vip_df.to_csv(top10_vip_path, index=False)

    print("\nTop 10 VIP wavelengths:")
    display(top10_vip_df)

    plot_top_vip_clean(
        vip_df=vip_df,
        top_n=10,
        output_path=vip_plot_path
    )

    print("\nVIP files saved:")
    print(vip_path)
    print(top10_vip_path)
    print(vip_plot_path)

else:
    print(
        f"\nBest model is {best_model_name}, not PLSR. "
        "VIP is only calculated for PLSR models."
    )


# ============================================================
# 8. Top 10 largest prediction errors
# ============================================================

original_oof_df.sort_values(
    "abs_error",
    ascending=False
).head(10).to_csv(
    FINAL_PLOT_DIR / "Top10_largest_errors_original_GroupKFold.csv",
    index=False
)

validation_plot_df.sort_values(
    "abs_error",
    ascending=False
).head(10).to_csv(
    FINAL_PLOT_DIR / "Top10_largest_errors_validation36.csv",
    index=False
)

print("\nAll clean figures and tables saved to:")
print(FINAL_PLOT_DIR)